<img src="http://imgur.com/1ZcRyrc.png" style="float: left; margin: 20px; height: 55px">

# `pandas` Long Format, Wide Format, Pivot Tables, and Melting
---


<br>

This lesson is all about transforming data using `pandas`. Data transformation is the reorganization of your data set's rows and columns into a different, potentially more useful shape and format. 

The benefits of transforming your data include better access to relevant information and streamlined data manipulation. As you become more familiar with data sets and their associated operations, you will develop an intuition and appreciation for when it's better to work row-wise or column-wise.

Different data formats are better for different tasks. It takes time and experience to learn the distinctions. But, for now, we'll introduce the common structures, transformations, and how to apply these transformations.

### Learning Objectives
- Understand the differences between long and wide format data.
- Understand pivot tables.
- Practice transforming data between long and wide formats.
- Practice creating pivot tables.
- Learn how to avoid common pitfalls and obstacles in data transformation with `pandas`.


### Lesson Guide

- [Wide Format Data](#wide_format)
- [Long Format Data](#long_format)
- [Using `pandas`' `.pivot()` Function: Long to Wide Format](#pivot)
- [Using `pandas`' `.melt()` Function: Wide to Long Format](#melt)



In [1]:
import numpy as np
import scipy.stats as stats
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
# pd.set_option('display.max_columns', None)
# sns.set_style('darkgrid')

# %config InlineBackend.figure_format = 'retina'
# %matplotlib inline

<a id='wide_format'></a>

### Wide Format Data

---

Between "wide" and "long," wide format data is the more intuitive. It's also a common format for `.csv` files. You've already viewed multiple data sets in wide format throughout this course.

Wide format data is structured so that:

- Unique IDs, subjects, observations, etc. are represented as rows.
- Distinct information categories (variables) are represented as columns. In other words, there is a column for every "variable" with its own unique values.
- This format can often be a more compact matrix, particularly if little or no information is missing.
- It is not as useful for SQL-style operations: It can make it much harder or even impossible to join tables together on a value.
- It can be useful in `pandas` when you need to perform operations on variables **across columns**; for example, multiplying columns together to create a new column.
- It is the data format required for statistical modeling (with few exceptions).


This data set is in a familiar format in which each column is a variable and each row contains an observation for that variable, corresponding to a distinct subject.

*Wide format implies that all of the information for one distinct subject will be represented in the columns corresponding to that row. A single subject should not be represented in multiple rows of data.*

<a id='load_nerdy'></a>

#### Load and Examine the "Life expectancy at birth" Data Set

---
You can find the raw data [here](https://databank.worldbank.org/source/world-development-indicators/Series/SP.DYN.LE00.IN#).





**Load the data (which is in wide format).** 

In [2]:
life = pd.read_csv("./data/Life_expectancy_at_birth.csv")
life.head()


,Country Name,Country Code,1990,2000,2016,2017,2018,2019,2020,2021,2022,2023,2024
0,Afghanistan,AFG,45.118,55.005,62.646,62.406,62.443,62.941,61.454,60.417,65.617,66.035,66.289
1,Albania,ALB,72.71,74.826,78.643,78.9,79.238,79.467,77.824,76.844,78.769,79.602,79.776
2,Algeria,DZA,67.658,70.576,75.31,75.431,75.555,75.682,73.257,75.208,76.129,76.261,76.475
3,American Samoa,ASM,71.107,71.443,72.64,72.801,72.794,72.751,72.672,72.708,72.752,72.852,72.992
4,Andorra,AND,78.961,81.863,84.489,84.359,84.242,84.098,79.418,82.331,84.016,84.041,84.188


In [3]:
# check shape
life.shape


(265, 13)

<a id='melt'></a>

### Using pandas' `.melt()` Function: Wide to Long Format

---

**`.melt()`** pd.melt() is a pandas function used to convert a dataframe from a wide format to a long format. is a function that essentially performs the inverse of `pivot()` on DataFrames.

`.melt()` takes a DataFrame as its first argument. Additional arguments typically used with this function are:

- **`id_vars`**: `Columns that stay locked in place`.
- **`value_vars`**: `Wide columns to collapse vertically`. A list that specifies which columns should be converted into single `value` and `variable` columns.
- **`var_name`**: `The header name for our new categorical column` (default='variable').
- **`value_name`**: `The header name for our numerical data column` (default='value').



In [13]:
# get the shape of the dataset
life.shape
                    



(265, 13)

In [15]:
# check the life expectency for Bahrain
life[life["Country Name"] == "Bahrain"]

,Country Name,Country Code,1990,2000,2016,2017,2018,2019,2020,2021,2022,2023,2024
14,Bahrain,BHR,72.587,74.673,80.407,80.407,80.412,80.471,78.683,78.083,80.992,81.284,81.423


In [44]:
# life.columns.to_list()[2:]

In [16]:
# convert the dataset from wide to long using pd.melt
life_long = pd.melt(frame = life,
                    id_vars = ["Country Name", "Country Code"],
                    value_vars = life.columns.to_list()[2:],
                    var_name = "Year",
                    value_name = "Life Expectancy")


# check the head of newly created dataframe                  
life_long.head()



,Country Name,Country Code,Year,Life Expectancy
0,Afghanistan,AFG,1990,45.118
1,Albania,ALB,1990,72.71
2,Algeria,DZA,1990,67.658
3,American Samoa,ASM,1990,71.107
4,Andorra,AND,1990,78.961


In [17]:
# check the life expectency for Bahrain in the new created long dataframe
life_long[life_long["Country Name"]== "Bahrain"]




,Country Name,Country Code,Year,Life Expectancy
14,Bahrain,BHR,1990,72.587
279,Bahrain,BHR,2000,74.673
544,Bahrain,BHR,2016,80.407
809,Bahrain,BHR,2017,80.407
1074,Bahrain,BHR,2018,80.412
1339,Bahrain,BHR,2019,80.471
1604,Bahrain,BHR,2020,78.683
1869,Bahrain,BHR,2021,78.083
2134,Bahrain,BHR,2022,80.992
2399,Bahrain,BHR,2023,81.284


**Saving the new long_life dataframe into a csv file**

In [18]:
# save data to csv as life_long
life_long.to_csv("./data/life_long.csv")


<a id='pivot'></a>

### `Pandas`' `.pivot()` Function: Long to Wide Format

---

The `pd.pivot()` function is a powerful tool for transforming data from long to wide format and it has an advanced version of it called `pd.pivot_table()` that can do summarizing of data with user-supplied functions.

First, we'll look at transforming the long format data back into the wide format using the `.pivot()` function.


**Important parameters for the `.pivot()` function include:**

> The `pivot()` function takes a DataFrame to pivot as its first argument. 
    
- **`columns`**: `Which column's values become the new headers?`. This is the list of columns in the long format data to be transformed back into columns in the wide format. After pivoting, each unique value in the long format column becomes a header in the wide format.
- **`values`**: `What data numbers fill the intersections?`. A single column indicating the values to use when pivoting and filling the new wide format columns.
- **`index`**: `What stays as your rows`. Columns in the long format data that are index variables. These will be left as single columns, not spread out by unique value like in the `columns` parameter.


In [19]:
# check the head of life_long
life_long.head()


,Country Name,Country Code,Year,Life Expectancy
0,Afghanistan,AFG,1990,45.118
1,Albania,ALB,1990,72.71
2,Algeria,DZA,1990,67.658
3,American Samoa,ASM,1990,71.107
4,Andorra,AND,1990,78.961


In [21]:
# convert the dataset from back wide to long using pd.pivot

life_wide = life_long.pivot(index = ["Country Name","Country Code"],
                            columns= "Year",
                            values= "Life Expectancy").reset_index()

# check the head of life_wide
life_wide.head()


Year,Country Name,Country Code,1990,2000,2016,2017,2018,2019,2020,2021,2022,2023,2024
0,Afghanistan,AFG,45.118,55.005,62.646,62.406,62.443,62.941,61.454,60.417,65.617,66.035,66.289
1,Africa Eastern and Southern,AFE,51.09632645,52.55734186,62.16798132,62.5912751,63.33069136,63.85726128,63.766484,62.97999919,64.48715179,65.14622792,65.34993047
2,Africa Western and Central,AFW,48.45115887,50.29260517,56.3875628,56.62076423,57.03077113,57.14282681,57.35770914,57.35501505,57.97899547,58.84748338,59.04153801
3,Albania,ALB,72.71,74.826,78.643,78.9,79.238,79.467,77.824,76.844,78.769,79.602,79.776
4,Algeria,DZA,67.658,70.576,75.31,75.431,75.555,75.682,73.257,75.208,76.129,76.261,76.475


**Remove the column label.**

Notice that in the above dataframe, "Year" is not a column but it is a name representing all the columns and it is created by default during the pivot function.
You can remove the column label (which can be confusing during print statements) by setting the `.columns.name` attribute to `None`.

In [22]:
# remove the column label
life_wide.columns.name = None
life_wide.head()


,Country Name,Country Code,1990,2000,2016,2017,2018,2019,2020,2021,2022,2023,2024
0,Afghanistan,AFG,45.118,55.005,62.646,62.406,62.443,62.941,61.454,60.417,65.617,66.035,66.289
1,Africa Eastern and Southern,AFE,51.09632645,52.55734186,62.16798132,62.5912751,63.33069136,63.85726128,63.766484,62.97999919,64.48715179,65.14622792,65.34993047
2,Africa Western and Central,AFW,48.45115887,50.29260517,56.3875628,56.62076423,57.03077113,57.14282681,57.35770914,57.35501505,57.97899547,58.84748338,59.04153801
3,Albania,ALB,72.71,74.826,78.643,78.9,79.238,79.467,77.824,76.844,78.769,79.602,79.776
4,Algeria,DZA,67.658,70.576,75.31,75.431,75.555,75.682,73.257,75.208,76.129,76.261,76.475


In [23]:
# check the life expectency for Bahrain in the new created wide dataframe

life_wide[life_wide['Country Name']=='Bahrain']


,Country Name,Country Code,1990,2000,2016,2017,2018,2019,2020,2021,2022,2023,2024
17,Bahrain,BHR,72.587,74.673,80.407,80.407,80.412,80.471,78.683,78.083,80.992,81.284,81.423
